# Step 4 — Target-rate threshold validation
Run Runtime → Run all, then save this notebook to GitHub. Standalone: earlier notebooks need not be rerun.

Goal: determine whether KXFED threshold quotes refine the open-ended decision categories. This is a data-quality diagnostic; no Treasury outcomes or regressions are used.

Sources: [historical markets](https://docs.kalshi.com/api-reference/historical/get-historical-markets), [historical candles](https://docs.kalshi.com/api-reference/historical/get-historical-market-candlesticks).

A greater-than threshold has probability P(R > k). Adjacent differences give P(a < R ≤ b). Neither finite intervals nor remaining tails automatically identify exact policy decisions or their variance. Quote candle age does not reveal actual quote-update age. Raw responses stay in memory; saving to GitHub preserves displayed audit tables.

Validation status: prepared for execution in Colab; live full-run outputs are pending.

### 1. Connect and reuse the eight fixed meeting cutoffs

In [ ]:
import time
import re
import math
import requests
import pandas as pd
from IPython.display import display

BASE = "https://external-api.kalshi.com/trade-api/v2"
session = requests.Session()
request_log = []
response_cache = {}

def get_json(path, params=None):
    params = dict(params or {})
    cache_key = (path, tuple(sorted(params.items())))
    if cache_key in response_cache:
        return response_cache[cache_key]
    for attempt in range(3):
        r = session.get(BASE + path, params=params, timeout=30)
        if r.status_code == 429 or r.status_code >= 500:
            if attempt < 2:
                try:
                    delay = float(r.headers.get("Retry-After", 2 ** (attempt + 1)))
                except ValueError:
                    delay = 2 ** (attempt + 1)
                time.sleep(min(max(delay, 1), 30))
                continue
        r.raise_for_status()
        result = r.json()
        request_log.append({
            "path": path, "params": params,
            "retrieved_utc": pd.Timestamp.now(tz="UTC").isoformat()
        })
        response_cache[cache_key] = result
        return result

# Explicit dates avoid treating contract closing times as official release times.
meeting_specs = [
    ("FEDDECISION-23MAY",   "2023-05-03", "2023-05-02"),
    ("FEDDECISION-23SEP",   "2023-09-20", "2023-09-19"),
    ("FEDDECISION-24JAN31", "2024-01-31", "2024-01-30"),
    ("FEDDECISION-24JUN",   "2024-06-12", "2024-06-11"),
    ("FEDDECISION-24SEP",   "2024-09-18", "2024-09-17"),
    ("KXFEDDECISION-24DEC", "2024-12-18", "2024-12-17"),
    ("KXFEDDECISION-25SEP", "2025-09-17", "2025-09-16"),
    ("KXFEDDECISION-26JUL", "2026-07-29", "2026-07-28"),
]
meetings = pd.DataFrame(meeting_specs, columns=["event", "meeting_date", "cutoff_date"])
meetings["cutoff_utc"] = meetings["cutoff_date"].map(
    lambda date: pd.Timestamp(date + " 16:00", tz="America/New_York").tz_convert("UTC")
)
assert meetings["meeting_date"].is_unique
assert len(meetings) == 8
display(meetings)


### 2. Find threshold markets and preserve their rules
Matches are candidates until the rule text confirms the meeting date, upper-bound target, units and strict inequality.

In [ ]:
market_rows = []
for path in ["/historical/markets", "/markets"]:
    cursor = None
    seen = set()
    for page in range(100):
        params = {"series_ticker": "KXFED", "limit": 100}
        if cursor:
            params["cursor"] = cursor
        result = get_json(path, params)
        market_rows.extend(result["markets"])
        cursor = result.get("cursor")
        if not cursor:
            break
        if cursor in seen:
            raise RuntimeError("Repeated cursor; inventory incomplete")
        seen.add(cursor)
    assert not cursor, "Pagination limit reached; inventory incomplete"

inventory = pd.DataFrame(market_rows).drop_duplicates("ticker")
# Preserve both legacy and current event prefixes; never infer meeting dates
# from closing timestamps or use eventual settlement values as predictors.
meetings["suffix"] = meetings["event"].str.replace(
    r"^(?:KX)?FEDDECISION-", "", regex=True
)
suffix_to_meeting = dict(zip(meetings["suffix"], meetings["meeting_date"]))
inventory["suffix"] = inventory["event_ticker"].str.replace(
    r"^(?:KX)?FED-", "", regex=True
)
selected = inventory.loc[inventory["suffix"].isin(suffix_to_meeting)].copy()
selected["meeting_date"] = selected["suffix"].map(suffix_to_meeting)
selected["threshold_pct"] = pd.to_numeric(selected["floor_strike"], errors="coerce")
selected["supported_greater_rule"] = (
    selected["strike_type"].eq("greater") & selected["threshold_pct"].notna()
)
display(selected.reindex(columns=[
    "meeting_date", "event_ticker", "ticker", "yes_sub_title",
    "strike_type", "floor_strike", "cap_strike", "supported_greater_rule"
]).sort_values(["meeting_date", "floor_strike"]))
print("Missing target-market matches:", sorted(
    set(meetings["meeting_date"]) - set(selected["meeting_date"])
))
# Print one rule per matched event to verify the target is the upper bound.
for event, group in selected.groupby("event_ticker"):
    market = group.iloc[0]
    print("\nRULE REVIEW:", event)
    print(market.get("title"), market.get("yes_sub_title"))
    print(market.get("rules_primary"))
    print(market.get("rules_secondary"))
print("Do not treat event-name matching alone as validated settlement alignment.")


### 3. Retrieve pre-cutoff quotes
Try one-minute candles near cutoff, then hourly candles over seven days. Missing markets and request failures stay visible. Allow several minutes.

In [ ]:
quote_rows = []
for _, meeting in meetings.iterrows():
    cutoff = meeting["cutoff_utc"]
    contracts = selected.loc[selected["meeting_date"].eq(meeting["meeting_date"])]
    print("Checking", meeting["meeting_date"], len(contracts), "thresholds", flush=True)
    for _, market in contracts.iterrows():
        ticker = market["ticker"]
        row = {
            "meeting_date": meeting["meeting_date"], "event": market["event_ticker"],
            "contract": ticker, "threshold_pct": market["threshold_pct"],
            "supported_greater_rule": bool(market["supported_greater_rule"]),
            "cutoff_utc": cutoff, "period_end_utc": pd.NaT,
            "bid": float("nan"), "ask": float("nan"), "midpoint": float("nan"),
            "period_age_hours": float("nan"), "quote_update_age": "unknown",
            "normalization": "none", "status": "pending"
        }
        try:
            candidates = []
            for interval, lookback in [(1, pd.Timedelta(minutes=15)),
                                       (60, pd.Timedelta(days=7))]:
                candles = get_json(f"/historical/markets/{ticker}/candlesticks", {
                    "start_ts": int((cutoff - lookback).timestamp()),
                    "end_ts": int(cutoff.timestamp()), "period_interval": interval
                })["candlesticks"]
                for candle in candles:
                    end = pd.to_datetime(candle["end_period_ts"], unit="s", utc=True)
                    bid = (candle.get("yes_bid") or {}).get("close")
                    ask = (candle.get("yes_ask") or {}).get("close")
                    if end <= cutoff and bid is not None and ask is not None:
                        candidates.append((end, float(bid), float(ask), candle))
                if candidates:
                    break
            if candidates:
                end, bid, ask, candle = max(candidates, key=lambda x: x[0])
                valid = math.isfinite(bid) and math.isfinite(ask) and 0 <= bid <= ask <= 1
                row.update({
                    "period_end_utc": end, "bid": bid, "ask": ask,
                    "midpoint": (bid + ask) / 2 if valid else float("nan"),
                    "period_age_hours": (cutoff - end).total_seconds() / 3600,
                    "spread": ask - bid, "period_interval_minutes": interval,
                    "carried_from_earlier_period": end < cutoff,
                    "period_volume": candle.get("volume"),
                    "status": "valid quote" if valid else "invalid quote"
                })
            else:
                row["status"] = "no two-sided candle in bounded search"
        except (requests.RequestException, ValueError, KeyError, TypeError) as exc:
            row["status"] = f"ERROR: {type(exc).__name__}: {exc}"
        quote_rows.append(row)

quote_audit = pd.DataFrame(quote_rows, columns=list(row) if quote_rows else [
    "meeting_date", "event", "contract", "threshold_pct", "supported_greater_rule",
    "bid", "ask", "midpoint", "period_age_hours", "status"
]) if not quote_rows else pd.DataFrame(quote_rows)
display(quote_audit)
if len(quote_audit):
    assert quote_audit["contract"].is_unique
    assert quote_audit["period_age_hours"].dropna().ge(0).all()


### 4. Check monotonicity, interval probabilities and unresolved tails
We compare thresholds only when their candle periods align. Negative raw differences are reported without repair. This diagnostic does not adopt a final stale-price rule.

In [ ]:
summary_rows = []
interval_rows = []
for _, meeting in meetings.iterrows():
    group = quote_audit.loc[quote_audit["meeting_date"].eq(meeting["meeting_date"])]
    good = group.loc[group["status"].eq("valid quote")].sort_values("threshold_pct")
    supported = good.loc[good["supported_greater_rule"].eq(True)]
    # Only a common candle period supports contemporaneous monotonicity checks.
    common_period = (len(supported) >= 2 and
                     supported["period_end_utc"].nunique() == 1)
    unique_thresholds = supported["threshold_pct"].is_unique
    midpoint_monotone = None
    quote_bands_feasible = None
    if common_period and unique_thresholds:
        points = supported.to_dict("records")
        midpoint_monotone = all(
            a["midpoint"] >= b["midpoint"] - 1e-9
            for a, b in zip(points, points[1:])
        )
        # There exists a non-increasing sequence inside all bid/ask bands iff
        # every earlier ask is at least every subsequent bid.
        quote_bands_feasible = all(
            a["ask"] >= b["bid"] - 1e-9
            for i, a in enumerate(points) for b in points[i+1:]
        )
        for a, b in zip(points, points[1:]):
            interval_rows.append({
                "meeting_date": meeting["meeting_date"],
                "rate_interval_pct": f"({a['threshold_pct']}, {b['threshold_pct']}]",
                "raw_midpoint_probability": a["midpoint"] - b["midpoint"],
                "band_lower": max(0, a["bid"] - b["ask"]),
                "band_upper": min(1, a["ask"] - b["bid"])
            })
    summary_rows.append({
        "meeting_date": meeting["meeting_date"], "listed_thresholds": len(group),
        "valid_quotes": len(good),
        "supported_greater_thresholds": len(supported),
        "common_quote_period": common_period,
        "unique_thresholds": unique_thresholds,
        "midpoints_monotone": midpoint_monotone,
        "monotone_sequence_within_quote_bands": quote_bands_feasible,
        "oldest_period_hours": good["period_age_hours"].max(),
        "lowest_threshold_pct": supported["threshold_pct"].min(),
        "highest_threshold_pct": supported["threshold_pct"].max(),
        "lower_tail_midpoint": (
            1 - supported.iloc[0]["midpoint"] if len(supported) else float("nan")),
        "upper_tail_midpoint": (
            supported.iloc[-1]["midpoint"] if len(supported) else float("nan")),
        "sigma_identified": False
    })
threshold_summary = pd.DataFrame(summary_rows)
display(threshold_summary)
if interval_rows:
    display(pd.DataFrame(interval_rows))
print("API requests:", len(request_log))
print("No quote normalization, monotonicity repair, tail closure, or zero filling applied.")
print("Threshold differences identify interval masses only when rules align.")
print("Rates inside intervals and beyond end thresholds remain unidentified.")
print("A 25-bp decision grid would be an explicit assumption, not an observed fact.")
print("Do not calculate sigma or declare feasibility PASS from these tables alone.")
print("Save to ResearchProject / main as 03_threshold_validation.ipynb.")
